In [ ]:
# ============================================================
# CELL 1 — Install & Import Libraries
# ============================================================

!pip install pyreadr --quiet
!pip install scikit-bio --quiet

import pyreadr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
from skbio.stats.ordination import pcoa as skbio_pcoa
from skbio import DistanceMatrix
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.1 MB/s eta 0:00:00
✅ All libraries imported successfully!


In [ ]:
# ============================================================
# CELL 2 — Load .RData File (Animesh's exact starting point)
# ============================================================
# The .RData file is what Franzosa's team provided to Animesh
# It is already partially pre-filtered — this is why Animesh
# starts with ~3,935 metabolites instead of 8,848 raw features
# pyreadr reads R data files directly into Python

print("=" * 50)
print("LOADING .RData FILE")
print("=" * 50)

# Load the .RData file
result = pyreadr.read_r('.RData')

# See what objects are stored inside
print(f"Objects found in .RData file:")
for i, key in enumerate(result.keys()):
    obj = result[key]
    print(f"  [{i+1}] '{key}' — type: {type(obj).__name__}", end="")
    if hasattr(obj, 'shape'):
        print(f", shape: {obj.shape}")
    else:
        print(f", value: {obj}")

LOADING .RData FILE
Objects found in .RData file:
  [1] 'metadata' — type: DataFrame, shape: (220, 13)
  [2] 'mtb' — type: DataFrame, shape: (220, 8849)
  [3] 'mtb.map' — type: DataFrame, shape: (8848, 10)
  [4] 'genera' — type: DataFrame, shape: (220, 11721)
  [5] 'species' — type: DataFrame, shape: (220, 55883)
  [6] 'genera.counts' — type: DataFrame, shape: (220, 11721)
  [7] 'species.counts' — type: DataFrame, shape: (220, 55883)


In [ ]:
# ============================================================
# CELL 3 — Extract Objects & Explore
# ============================================================

# Extract each object from the .RData file
metadata = result['metadata']
mtb = result['mtb']
mtb_map = result['mtb.map']
genera = result['genera']
species = result['species']

print("=" * 50)
print("METABOLITE DATA (mtb)")
print("=" * 50)
print(f"Shape: {mtb.shape}")
print(f"First 3 rows, first 3 columns:")
print(mtb.iloc[:3, :3])

print("\n" + "=" * 50)
print("METADATA")
print("=" * 50)
print(f"Shape: {metadata.shape}")
print(f"Columns: {metadata.columns.tolist()}")
print(f"\nDiagnosis distribution:")
print(metadata['Study.Group'].value_counts())

print("\n" + "=" * 50)
print("MICROBIAL DATA")
print("=" * 50)
print(f"Genera shape: {genera.shape}")
print(f"Species shape: {species.shape}")
print(f"\nFirst 3 genera columns:")
print(genera.columns[:3].tolist())

print("\n" + "=" * 50)
print("KEY DIFFERENCE FROM NOTEBOOK 1")
print("=" * 50)
print(f"Notebook 1 (raw mtb.tsv): 8,848 metabolites")
print(f"Notebook 2 (.RData mtb):  {mtb.shape[1]} metabolites")
print(f"Difference: {mtb.shape[1] - 8848} metabolites")
print(f"\n✅ .RData contains SAME raw data + microbial data!")
print(f"Animesh's filtering happens in his R code")

METABOLITE DATA (mtb)
Shape: (220, 8849)
First 3 rows, first 3 columns:
       Sample  C18-neg_Cluster_0001: NA  C18-neg_Cluster_0002: NA
0  PRISM.7122                      0.00                 6391.0100
1  PRISM.7147                   1635.54                   27.4461
2  PRISM.7150                      0.00                 8265.9000

METADATA
Shape: (220, 13)
Columns: ['Dataset', 'Sample', 'Subject', 'Study.Group', 'Age', 'Age.Units', 'DOI', 'Publication.Name', 'Fecal.Calprotectin', 'antibiotic', 'immunosuppressant', 'mesalamine', 'steroids']

Diagnosis distribution:
Study.Group
CD         88
UC         76
Control    56
Name: count, dtype: int64

MICROBIAL DATA
Genera shape: (220, 11721)
Species shape: (220, 55883)

First 3 genera columns:
['Sample', 'd__Bacteria;p__Firmicutes_A;c__Clostridia;o__Peptostreptococcales;f__Acidaminobacteraceae;g__Fusibacter_A', 'd__Bacteria;p__Firmicutes_A;c__Clostridia;o__Oscillospirales;f__CAG-272;g__Flemingibacterium']

KEY DIFFERENCE FROM NOTEBOOK 1
N

In [ ]:
# ============================================================
# CELL 4 — Set Up Data Properly
# ============================================================

print("=" * 50)
print("SETTING UP DATA")
print("=" * 50)

# Set Sample as index for mtb
mtb_data = mtb.set_index('Sample')
print(f"mtb shape after setting index: {mtb_data.shape}")

# Set Sample as index for metadata
metadata_data = metadata.set_index('Sample')
print(f"metadata shape: {metadata_data.shape}")

# Set Sample as index for genera
genera_data = genera.set_index('Sample')
print(f"genera shape: {genera_data.shape}")

# Verify alignment
common = mtb_data.index.intersection(metadata_data.index)
print(f"\nCommon samples: {len(common)}")

# Align all datasets
mtb_data = mtb_data.loc[common]
metadata_data = metadata_data.loc[common]
genera_data = genera_data.loc[common]

print(f"\n✅ All datasets aligned!")
print(f"Patients: {len(common)}")
print(f"Metabolites: {mtb_data.shape[1]:,}")
print(f"Microbial genera: {genera_data.shape[1]:,}")

# Extract diagnosis labels
labels = metadata_data['Study.Group']
print(f"\nDiagnosis distribution:")
print(labels.value_counts())

# This is Animesh's EXACT starting point
print(f"\n✅ This is Animesh's exact starting point!")
print(f"Same data, same patients, same metabolites")
print(f"Now we apply his filtering steps...")

SETTING UP DATA
mtb shape after setting index: (220, 8848)
metadata shape: (220, 12)
genera shape: (220, 11720)

Common samples: 220

✅ All datasets aligned!
Patients: 220
Metabolites: 8,848
Microbial genera: 11,720

Diagnosis distribution:
Study.Group
CD         88
UC         76
Control    56
Name: count, dtype: int64

✅ This is Animesh's exact starting point!
Same data, same patients, same metabolites
Now we apply his filtering steps...


In [ ]:
# ============================================================
# CELL 5 — Step 1: Sparsity Filtering
# ============================================================
# Animesh's R code:
# sparsity_mtb <- c(rep(0,8849))
# for (i in 1:ncol(mtb)) {
#     sparsity_mtb[i] <- (sum(mtb[, i] == 0)/220)*100
# }
# remove_mtb <- which(sparsity_mtb > 80)
# mtb_alt <- mtb[-c(remove_mtb)]
#
# Remove metabolites where MORE than 80% of patients
# have a zero value (metabolite was not detected)
# These are too sparse to be useful for classification

print("=" * 50)
print("STEP 1: SPARSITY FILTERING")
print("=" * 50)
print("Logic: remove metabolites where >80% of patients = 0")

# Calculate sparsity % for each metabolite
sparsity = (mtb_data == 0).sum(axis=0) / len(mtb_data) * 100

# Find metabolites to remove
remove = sparsity[sparsity > 80].index

# Apply filter
mtb_alt = mtb_data.drop(columns=remove)

print(f"\nOriginal metabolites: {mtb_data.shape[1]:,}")
print(f"Removed (>80% zeros): {len(remove):,}")
print(f"Remaining: {mtb_alt.shape[1]:,}")
print(f"\n✅ mtb_alt shape: {mtb_alt.shape}")

STEP 1: SPARSITY FILTERING
Logic: remove metabolites where >80% of patients = 0

Original metabolites: 8,848
Removed (>80% zeros): 13
Remaining: 8,835

✅ mtb_alt shape: (220, 8835)


In [ ]:
# ============================================================
# CELL 6 — Step 2: CLR Transformation
# ============================================================
# Animesh's R code:
# tran_mtb <- clr(mtb_alt + 1)
#
# CLR = Centred Log Ratio
# Metabolomics data is compositional - values are relative
# CLR makes samples comparable by normalising against
# each sample's own geometric mean
# +1 pseudo-count added to avoid log(0) errors

print("=" * 50)
print("STEP 2: CLR TRANSFORMATION")
print("=" * 50)

# Add pseudo-count of 1
mtb_pseudo = mtb_alt + 1
print(f"Pseudo-count of 1 added")
print(f"Zeros before: {(mtb_alt == 0).sum().sum():,}")
print(f"Zeros after: {(mtb_pseudo == 0).sum().sum():,}")

# Apply CLR transform
mtb_clr_arr = clr(mtb_pseudo.values)
tran_mtb = pd.DataFrame(
    mtb_clr_arr,
    index=mtb_alt.index,
    columns=mtb_alt.columns
)

print(f"\n✅ CLR complete!")
print(f"Mean (should be ~0): {tran_mtb.values.mean():.6f}")
print(f"Shape: {tran_mtb.shape}")

STEP 2: CLR TRANSFORMATION
Pseudo-count of 1 added
Zeros before: 528,330
Zeros after: 0

✅ CLR complete!
Mean (should be ~0): -0.000000
Shape: (220, 8835)


In [ ]:
# ============================================================
# CELL 7 — Step 3: Z-Score Normalisation
# ============================================================
# Animesh's R code:
# mtb_norm <- as.data.frame(scale(tran_mtb))
#
# Z-score makes every metabolite have:
# Mean = 0 and Standard Deviation = 1
# Formula: Z = (value - mean) / std
# Prevents large-valued metabolites from dominating ML models

print("=" * 50)
print("STEP 3: Z-SCORE NORMALISATION")
print("=" * 50)

scaler = StandardScaler()
mtb_norm = pd.DataFrame(
    scaler.fit_transform(tran_mtb),
    index=tran_mtb.index,
    columns=tran_mtb.columns
)

print(f"✅ Z-score complete!")
print(f"Mean (should be ~0): {mtb_norm.values.mean():.6f}")
print(f"Std (should be ~1):  {mtb_norm.values.std():.6f}")
print(f"Shape: {mtb_norm.shape}")

STEP 3: Z-SCORE NORMALISATION
✅ Z-score complete!
Mean (should be ~0): 0.000000
Std (should be ~1):  1.000000
Shape: (220, 8835)


In [ ]:
# ============================================================
# CELL 8 — Step 4: Outlier Removal
# ============================================================
# Animesh's R code:
# met_diff <- mtb_alt[-c(126,182),]
# metadata_diff <- metadata[-c(126,182),]
#
# Animesh identified 2 outlier patients from his PCoA plot
# on the MICROBIOME (genera) data - not metabolomics!
# He hardcoded rows 126 and 182 in R
# R rows 126, 182 = Python rows 125, 181 (0-indexed)

print("=" * 50)
print("STEP 4: OUTLIER REMOVAL")
print("=" * 50)
print("Animesh hardcoded rows 126 & 182 (R) = 125 & 181 (Python)")

# Identify outlier patients
outlier_patients = [mtb_norm.index[125], mtb_norm.index[181]]
print(f"\nOutlier patients identified:")
for p in outlier_patients:
    print(f"  {p}")

# Remove from metabolite data
met_diff = mtb_norm.drop(index=outlier_patients)

# Remove from metadata
metadata_diff = metadata_data.drop(index=outlier_patients)

# Add Disease.Group column (IBD vs Control)
metadata_diff = metadata_diff.copy()
metadata_diff['Disease.Group'] = metadata_diff['Study.Group'].apply(
    lambda x: 'Control' if x == 'Control' else 'IBD'
)

print(f"\nPatients before: {mtb_norm.shape[0]}")
print(f"Patients removed: {len(outlier_patients)}")
print(f"Patients after: {met_diff.shape[0]}")
print(f"Expected: 218 ✅")
print(f"\nDiagnosis distribution:")
print(metadata_diff['Study.Group'].value_counts())

STEP 4: OUTLIER REMOVAL
Animesh hardcoded rows 126 & 182 (R) = 125 & 181 (Python)

Outlier patients identified:
  PRISM.8802
  Validation.UMCG1708847

Patients before: 220
Patients removed: 2
Patients after: 218
Expected: 218 ✅

Diagnosis distribution:
Study.Group
CD         87
UC         75
Control    56
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 9 — Preprocessing Step 5: Mann-Whitney + FDR Filtering
# ============================================================
# Animesh's R code:
# Uses Wilcoxon test (= Mann-Whitney) + Hochberg correction
# Keeps only metabolites significantly different
# between IBD and Control (FDR < 0.05)

print("=" * 50)
print("STEP 5: MANN-WHITNEY TEST + FDR FILTERING")
print("=" * 50)

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Get IBD and Control indices
ibd_idx = metadata_clean[
    metadata_clean['Disease.Group'] == 'IBD'
].index
ctrl_idx = metadata_clean[
    metadata_clean['Disease.Group'] == 'Control'
].index

print(f"IBD patients: {len(ibd_idx)}")
print(f"Control patients: {len(ctrl_idx)}")
print(f"Metabolites to test: {mtb_clean.shape[1]:,}")
print(f"\nRunning Mann-Whitney U test...")

# Test each metabolite
p_values = []
for col in mtb_clean.columns:
    _, p = mannwhitneyu(
        mtb_clean.loc[ibd_idx, col],
        mtb_clean.loc[ctrl_idx, col],
        alternative='two-sided'
    )
    p_values.append(p)

# Apply BH-FDR correction
reject, p_adj, _, _ = multipletests(
    p_values, method='fdr_bh', alpha=0.05
)

# Keep only significant metabolites
sig_cols = mtb_clean.columns[reject]
mtb_significant = mtb_clean[sig_cols]

print(f"\n✅ Significant metabolites (FDR<0.05): {len(sig_cols):,}")
print(f"Expected from Animesh: ~897")
print(f"\n✅ Final dataset: {mtb_significant.shape[0]} patients × "
      f"{mtb_significant.shape[1]:,} metabolites")

STEP 5: MANN-WHITNEY TEST + FDR FILTERING
IBD patients: 162
Control patients: 56
Metabolites to test: 8,835

Running Mann-Whitney U test...

✅ Significant metabolites (FDR<0.05): 5,397
Expected from Animesh: ~897

✅ Final dataset: 218 patients × 5,397 metabolites


In [ ]:
# ============================================================
# CELL 10 — Pipeline Summary & Comparison with Animesh
# ============================================================

print("=" * 60)
print("NOTEBOOK 2 — PREPROCESSING PIPELINE SUMMARY")
print("(.RData starting point — Animesh's exact method)")
print("=" * 60)

print("""
STEP | DESCRIPTION              | OUR RESULT | ANIMESH
-----|--------------------------|------------|--------
  1  | Raw metabolites (.RData) | 8,848      | 8,849
  2  | After sparsity filter    | 8,835      | ~8,835
  3  | After unknown removal*   | 8,835      | ~3,935
  4  | After CLR transform      | 8,835      | ~3,935
  5  | After Z-score            | 8,835      | ~3,935
  6  | After outlier removal    | 218 ✅     | 218 ✅
  7  | After Mann-Whitney+FDR   | 5,397      | ~897

* Animesh removed metabolites with no HMDB/KEGG/Name/Class
  Dr. Guellil advised KEEPING these as HMDB is publicly
  available and these features carry real biological signal
  (confirmed by our heatmap: top 20 significant = all unknown)
""")

print("KEY FINDING:")
print("=" * 60)
print(f"""
✅ Patient count matches EXACTLY: 218 patients
✅ Outlier patients identified: PRISM.8802 & Validation.UMCG1708847
✅ CLR transform: mean = 0.000000
✅ Z-score: mean = 0, std = 1

⚠️  Metabolite count differs (5,397 vs 897) because:
   Dr. Guellil advised NOT removing unknown metabolites
   since HMDB is publicly available for annotation.
   Our 5,397 includes ALL potentially significant features.
   Animesh's 897 comes from first removing ~4,900 unknowns.

🔬 BIOLOGICAL INSIGHT:
   Among our 5,397 significant metabolites:
   - Many are unidentified Cluster_XXXX features
   - These unknowns carry real disease signal
   - Removing them (as Animesh did) loses information
   - Future work: annotate using HMDB API
""")

# Save final dataset
mtb_significant.to_csv('franzosa_preprocessed_RData.csv')
metadata_clean.to_csv('franzosa_metadata_RData.csv')
print("✅ Data saved!")
print("🎉 NOTEBOOK 2 COMPLETE!")

NOTEBOOK 2 — PREPROCESSING PIPELINE SUMMARY
(.RData starting point — Animesh's exact method)

STEP | DESCRIPTION              | OUR RESULT | ANIMESH
-----|--------------------------|------------|--------
  1  | Raw metabolites (.RData) | 8,848      | 8,849
  2  | After sparsity filter    | 8,835      | ~8,835
  3  | After unknown removal*   | 8,835      | ~3,935
  4  | After CLR transform      | 8,835      | ~3,935
  5  | After Z-score            | 8,835      | ~3,935
  6  | After outlier removal    | 218 ✅     | 218 ✅
  7  | After Mann-Whitney+FDR   | 5,397      | ~897

* Animesh removed metabolites with no HMDB/KEGG/Name/Class
  Dr. Guellil advised KEEPING these as HMDB is publicly 
  available and these features carry real biological signal
  (confirmed by our heatmap: top 20 significant = all unknown)

KEY FINDING:

✅ Patient count matches EXACTLY: 218 patients
✅ Outlier patients identified: PRISM.8802 & Validation.UMCG1708847  
✅ CLR transform: mean = 0.000000
✅ Z-score: mean = 0,